In [1]:
from dotenv import find_dotenv, load_dotenv

In [2]:
from pydantic import BaseModel, Field, ValidationError
from openai import AzureOpenAI
from enum import Enum
from typing import List, Optional
from openai import BadRequestError, APITimeoutError, RateLimitError
import os
import time

In [3]:
from constants import cot_ner_prompt

In [4]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)


In [5]:
if find_dotenv():
    load_dotenv()

In [6]:
client = AzureOpenAI(
  azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT"), 
  api_key=os.getenv("azure_key"),  
  api_version=os.getenv("AZURE_API_VERSION")
)

In [7]:
class Label_Types(Enum):
    B_Object: str = "B-Object"
    I_Object: str = "I-Object"
    B_Start_location: str = "B-Start_location"
    I_Start_location: str = "I-Start_location"
    B_End_location: str = "B-End_location"
    I_End_location: str = "I-End_location"
    O: str = "O"  

class Label(BaseModel):
    label: Label_Types = Field(..., description="The set of label types to be used in this task")
    entity: Optional[str] = None
    
class TrainingPair(BaseModel):
    sentence: str = Field(..., description="The target text that you will be using to extract recognize entities")
    labels: List[Label] = Field(..., description="The output list of labels. Assign labels to all words, not just entities. I want to see them O for words that are not entities!")
    words: List[str] = Field(..., description="The output list of words/tokens. They should be equal to the number of labels!")

class Dataset(BaseModel):
    dataset: List[TrainingPair] = Field(..., description="The list of extracted and formatted text-entity dataset.")


# completion = client.beta.chat.completions.parse(
#     model="gpt-4o-test", 
#     messages=[
#         {"role": "system", "content": base_system_prompt},
#         {"role": "user", "content": examples},
#     ],
#     response_format=Dataset,
# )

# generated_dataset = completion.choices[0].message.parsed
# dataset = Dataset.model_validate(generated_dataset)

In [8]:
def parse_with_retry(
    client: AzureOpenAI,
    model: str,
    messages: List[dict],
    dataset_response_format: BaseModel,
    max_retries: int = 1,
    initial_delay: float = 1.0,
    backoff_factor: float = 2.0
) -> Optional[BaseModel]:
    """
    Attempts to parse the API call with retries on specific exceptions.
    
    Args:
        client: The OpenAI client instance.
        model (str): The model name to use for the API call.
        messages (List[dict]): The list of messages to send to the API.
        response_format (BaseModel): The Pydantic model to parse the response into.
        max_retries (int): Maximum number of retry attempts.
        initial_delay (float): Initial delay between retries in seconds.
        backoff_factor (float): Factor by which the delay increases after each retry.
        
    Returns:
        Optional[BaseModel]: The parsed response or None if all retries fail.
    """
    attempt = 0
    delay = initial_delay
    
    while attempt < max_retries:
        try:
            logging.info("Attempt %d: Making API call to model", attempt + 1)
            completion = client.beta.chat.completions.parse(
                model=model, 
                messages=messages,
                response_format=dataset_response_format,
                seed=42
            )
            logging.info(f"Messages {messages}")
            generated_dataset = completion.choices[0].message.parsed
            dataset = dataset_response_format.model_validate(generated_dataset)
            logging.info(f"API call successful on attempt {attempt+1} with model {model}")
            return dataset
        
        except BadRequestError as e:
            logging.error("BadRequestError: %s", e)
            break
        
        except RateLimitError as e:
            attempt += 1
            logging.warning("RateLimitError: %s. Retrying in %.1f seconds...", e, delay)
            time.sleep(delay)
            delay *= backoff_factor
        
        except APITimeoutError as e:
            attempt += 1
            logging.warning("APITimeoutError: %s. Retrying in %.1f seconds...", e, delay)
            time.sleep(delay)
            delay *= backoff_factor
        
        except Exception as e:
            attempt += 1
            logging.warning("Unexpected error: %s. Retrying in %.1f seconds...", e, delay)
            time.sleep(delay)
            delay *= backoff_factor
    
    logging.error("All %d attempts failed. Unable to parse the dataset.", max_retries)
    return None


In [9]:
def generate_dataset(number_datapoints: int, model: str = "gpt-4o-mini"):
    """
    Generates a dataset with N datapoints for NER task.
    
    Args:
        number_datapoints (int): Number of datapoints to generate.
        model (str): Azure model to use.
        
    """
    prompt = cot_ner_prompt.format(number_datapoints=number_datapoints)
    
    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": f"Please create {number_datapoints} datapoints"},
    ]
    
    dataset = parse_with_retry(
        client=client,
        model=model,
        messages=messages,
        dataset_response_format=Dataset,
        max_retries=5,
        initial_delay=1.0,
        backoff_factor=2.0
    )
    
    return dataset

In [10]:
training_data = generate_dataset(number_datapoints=3)

2024-11-26 17:25:44,187 [INFO] Attempt 1: Making API call to model
2024-11-26 17:25:50,156 [INFO] HTTP Request: POST https://finetuning-region.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2024-09-01-preview "HTTP/1.1 200 OK"
2024-11-26 17:25:50,179 [INFO] Messages [{'role': 'system', 'content': '"\nYou are a data annotator tasked with generating a dataset for a Named Entity Recognition (NER) task that will be used for \ntraining a mobile home assistant robot that will be able to perform basic home assistance tasks \nsuch as grabbing and moving objects. The robot through an interface should be able to receive commands, and use the NER model\nto recognize entities and their position in the house. \nMake sure to produce data that follows the IOB/BIO annotation format.\nFor each datapoint, perform the following steps:\n\n1. **Thought**: Reason and create sentence to identify potential entities and locations.\n2. **Action**: Decide on an action to identify en

In [16]:
from pprint import pprint
import json
dataset=Dataset.model_dump_json(training_data)
dataset

'{"dataset":[{"sentence":"Move the book from the living room to the study.","labels":[{"label":"B-Object","entity":"book"},{"label":"I-Object","entity":"null"},{"label":"B-Start_location","entity":"living room"},{"label":"I-Start_location","entity":"null"},{"label":"B-End_location","entity":"study"},{"label":"I-End_location","entity":"null"}],"words":["Move","the","book","from","the","living","room","to","the","study."]},{"sentence":"Grab the mug in the kitchen and take it to the dining room.","labels":[{"label":"B-Object","entity":"mug"},{"label":"I-Object","entity":"null"},{"label":"B-Start_location","entity":"kitchen"},{"label":"I-Start_location","entity":"null"},{"label":"B-End_location","entity":"dining room"},{"label":"I-End_location","entity":"null"}],"words":["Grab","the","mug","in","the","kitchen","and","take","it","to","the","dining","room."]},{"sentence":"Put the remote on the coffee table in the living room.","labels":[{"label":"B-Object","entity":"remote"},{"label":"I-Obje